# 03 · Sintaxis y entidades

## Propósito

Pasar de la palabra a la oración. Ver cómo spaCy conecta los tokens entre sí,
cómo agrupa las frases nominales y cómo detecta nombres propios.

Al final vamos a rehacer, con estas herramientas, el conteo de palabras clave de
la clase de text mining, y comparar los dos resultados.

## Qué consume

- `datos/corpus.jsonl`

## ◈ Microglosario

| Término | Qué es |
|---|---|
| **dependencia** | La relación entre dos tokens: cuál depende de cuál y con qué función |
| **cabeza** (*head*) | El token del que depende otro. El verbo suele ser cabeza de su sujeto |
| **grupo nominal** (*noun chunk*) | Un sustantivo con lo que lo acompaña: *la vecina de al lado* |
| **entidad nombrada** | Un `Span` que el modelo clasificó como nombre propio de algo |
| **`PER`, `LOC`, `ORG`, `MISC`** | Las cuatro etiquetas de entidad que maneja este modelo: persona, lugar, organización y un cajón de sastre |

## Celda de reanudación

In [ ]:
# Recarga automática si editamos archivos externos (.py)
%load_ext autoreload
%autoreload 2

# spacy: la librería de análisis lingüístico.
import spacy

# displacy: el visualizador que viene adentro de spaCy.
from spacy import displacy

# pandas: para trabajar el corpus como una tabla.
import pandas as pd

# Counter: para contar.
from collections import Counter

# es_core_news_lg: modelo en español (es), entrenado con noticias (news), grande (lg)
nlp = spacy.load("es_core_news_lg")
# lines=True: el archivo .jsonl tiene un registro JSON por renglón
corpus = pd.read_json("datos/corpus.jsonl", lines=True)

cantidad_parrafos = len(corpus)
# :, separa los miles con una coma
print("párrafos:", f"{cantidad_parrafos:,}")

# pipe_names: la lista de componentes que tiene el pipeline cargado
if "parser" not in nlp.pipe_names:
    print("ATENCIÓN: sin parser no hay árbol sintáctico: ni doc.sents, ni noun_chunks, ni dependencias.")
    print("Todo el cuaderno depende de él.")

## 1. Dependencias sintácticas

Hasta ahora miramos cada token por separado. El `parser` hace algo distinto:
conecta cada token con otro del que depende.

El resultado es un árbol. Cada token tiene un **padre** (`token.head`) y una
**etiqueta** que dice qué relación tiene con él (`token.dep_`).

In [ ]:
oracion = nlp("La vecina de Mercedes le regaló una torta a mi hermana.")

filas = []
for token in oracion:
    # spacy.explain(): traduce la etiqueta a una descripción
    explicacion = spacy.explain(token.dep_)
    fila = {
        "token": token.text,
        "relacion": token.dep_,  # dep_: la relación sintáctica con la palabra de la que depende
        "depende_de": token.head.text,  # head: la palabra de la que depende
        "que_es_esa_relacion": explicacion,
    }
    filas.append(fila)

pd.DataFrame(filas)

Las relaciones que más vas a usar:

| etiqueta | qué marca |
|---|---|
| `ROOT` | El núcleo de la oración. Casi siempre el verbo principal. |
| `nsubj` | El sujeto: **quién** hace la acción. |
| `obj` | El objeto directo: **qué** recibe la acción. |
| `obl` | Un complemento: a quién, dónde, cuándo. |
| `amod` | Un adjetivo que modifica a un sustantivo. |
| `det` | Un determinante (el, la, un). |

Se ve mejor dibujado.

In [ ]:
# displacy.render(): dibuja el análisis. style="dep": el árbol de dependencias.
# distance: cuánto espacio deja entre palabra y palabra.
displacy.render(oracion, style="dep", jupyter=True, options={"distance": 110})

### Por qué esto importa para analizar textos

`nsubj` y `obj` no son curiosidades gramaticales: son **quién actúa** y **quién
recibe la acción**. En un corpus de noticias, la diferencia entre

> *La policía detuvo a los manifestantes*

y

> *Hubo detenciones*

es exactamente esto, y es contable.

Veamos cómo se extraen los sujetos de un párrafo del corpus.

In [ ]:
# .loc[2, "texto"]: el texto de la fila 2
parrafo = corpus.loc[2, "texto"]
doc = nlp(parrafo)

filas = []
for token in doc:

    # Solo nos interesan los tokens que son sujeto de algún verbo (nsubj: sujeto nominal).
    if token.dep_ == "nsubj":
        fila = {"sujeto": token.text, "verbo": token.head.text}
        filas.append(fila)

pd.DataFrame(filas)

### Recorrer el árbol

Desde cualquier token se puede subir al padre (`token.head`) o bajar a los hijos
(`token.children`). Eso permite preguntas como "¿con qué adjetivos aparece esta
palabra?".

Mirá bien el resultado de la celda siguiente antes de seguir.

In [ ]:
prueba = nlp("Era una casa vieja y húmeda, con un patio enorme y descuidado.")

for token in prueba:

    # Solo nos interesan los sustantivos.
    if token.pos_ == "NOUN":
        adjetivos = []
        # children: las palabras que dependen directamente de este token
        for hijo in token.children:
            if hijo.pos_ == "ADJ":
                adjetivos.append(hijo.text)
        if adjetivos:
            # join(): une la lista en un solo texto, separado por ", "
            adjetivos_como_texto = ", ".join(adjetivos)
            print(" ", token.text, "->", adjetivos_como_texto)

**Faltan la mitad de los adjetivos.** El texto decía *vieja y húmeda* y
*enorme y descuidado*, y solo aparecieron `vieja` y `enorme`.

No es un error del modelo: es cómo funciona el árbol. En una coordinación, el
segundo adjetivo **no** cuelga del sustantivo sino del primer adjetivo, con la
relación `conj`. Comprobalo.

In [ ]:
for token in prueba:

    # Solo nos interesan los adjetivos.
    if token.pos_ == "ADJ":
        hijos = []
        for hijo in token.children:
            descripcion = hijo.text + " (" + hijo.dep_ + ")"
            hijos.append(descripcion)
        print(" ", token.text, "-> hijos:", hijos)

Para juntarlos todos hay que bajar un nivel más.

In [ ]:
for token in prueba:

    # Solo nos interesan los sustantivos.
    if token.pos_ == "NOUN":
        adjetivos = []
        for hijo in token.children:

            # Y de los hijos, solo los que son adjetivos.
            if hijo.pos_ == "ADJ":
                adjetivos.append(hijo.text)

                # Y ahora los que cuelgan de ese adjetivo por coordinación (conj).
                for nieto in hijo.children:
                    if nieto.dep_ == "conj":
                        adjetivos.append(nieto.text)

        if adjetivos:
            adjetivos_como_texto = ", ".join(adjetivos)
            print(" ", token.text, "->", adjetivos_como_texto)

Guardate esto: **la primera versión de una consulta sintáctica casi nunca
captura todo.** Si hubieras contado adjetivos con el código anterior, te habrías
perdido la mitad sin enterarte. La forma de darte cuenta es la de siempre: mirar
un caso a mano y compararlo con lo que devolvió el código.

## 2. Grupos nominales

Un *noun chunk* es un sustantivo con lo que lo acompaña: determinantes y
adjetivos. spaCy los arma solo, a partir del árbol sintáctico.

In [ ]:
doc_chunks = nlp("La vieja radio de onda corta tenía una antena altísima.")

filas = []
# noun_chunks: los grupos nominales del documento, uno por uno
for grupo in doc_chunks.noun_chunks:
    # root: la palabra núcleo del grupo
    fila = {"grupo_nominal": grupo.text, "nucleo": grupo.root.text}
    filas.append(fila)

pd.DataFrame(filas)

Fijate que cada grupo es un `Span`, igual que las oraciones del cuaderno `01`.

Los grupos nominales son útiles cuando te interesa *de qué* habla un texto, más
que qué palabras usa: `la vieja radio de onda corta` dice más que `radio`.

## 3. Entidades nombradas

El componente `ner` marca nombres propios y los clasifica. El modelo español
usa cuatro etiquetas.

In [ ]:
noticia = nlp("Hernán Casciari fundó Editorial Orsai en Buenos Aires después de vivir en Barcelona.")

filas = []
# ents: las entidades nombradas que encontró el modelo
for entidad in noticia.ents:
    explicacion = spacy.explain(entidad.label_)
    fila = {
        "entidad": entidad.text,
        "tipo": entidad.label_,  # label_: el tipo de entidad (PER, LOC, ORG, MISC)
        "que_significa": explicacion,
    }
    filas.append(fila)

pd.DataFrame(filas)

| etiqueta | qué es |
|---|---|
| `PER` | Persona |
| `LOC` | Lugar |
| `ORG` | Organización |
| `MISC` | Otros nombres propios que no entran en las anteriores |

Y ya en este ejemplo de cuatro entidades hay un error: **`Editorial Orsai` es una
organización y el modelo la marcó como lugar.** Dejalo anotado; el cuaderno `04`
se ocupa de esto.

Las entidades también son `Span`, así que conservan su posición en el texto.

In [ ]:
# start y end: dónde empieza y dónde termina la entidad, contado en tokens
for entidad in noticia.ents:
    print(" ", entidad.text, "-> tokens", entidad.start, "a", entidad.end)

In [ ]:
# style="ent": marca las entidades con colores, adentro del texto
displacy.render(noticia, style="ent", jupyter=True)

### Entidades a tabla

Para analizar hace falta salir del `Doc` y entrar a una tabla. Este paso es el
que convierte texto en datos.

In [ ]:
primeros = corpus.head(400)
muestra = list(primeros["texto"])

registros = []
# batch_size=50: spaCy procesa los textos de a 50 por vez
for documento in nlp.pipe(muestra, batch_size=50):
    for entidad in documento.ents:
        registro = {"texto": entidad.text, "tipo": entidad.label_}
        registros.append(registro)

entidades = pd.DataFrame(registros)

cantidad_entidades = len(entidades)
print("entidades encontradas:", f"{cantidad_entidades:,}")
print()
# value_counts(): cuántas veces aparece cada valor de la columna
conteo_por_tipo = entidades["tipo"].value_counts()
print(conteo_por_tipo)

In [ ]:
# Las personas más mencionadas en la muestra.
# Para cada fila, ¿es de tipo PER? Da True o False. Nos quedamos con las True.
es_persona = entidades["tipo"] == "PER"
personas = entidades[es_persona]

mas_frecuentes = personas["texto"].value_counts()
quince_mas_frecuentes = mas_frecuentes.head(15)
print(quince_mas_frecuentes)

Mirá esa lista con atención antes de seguir. **Hay cosas que no son personas.**

No las vamos a arreglar todavía: el cuaderno `04` está dedicado entero a
auditar estos errores y entender de dónde salen. Por ahora, quedate con la
sensación de que la salida hay que mirarla.

## 4. Palabras clave: la misma pregunta de text mining, con otras herramientas

En la clase anterior contaste las palabras más frecuentes del corpus. El
procedimiento fue: pasar a minúsculas, sacar puntuación, filtrar con una lista
de palabras vacías armada a mano.

Hagamos las dos cosas sobre el mismo texto y comparemos.

In [ ]:
# --- Camino 1: como en text mining ---

# re: expresiones regulares.
import re

# string: trae la lista de signos de puntuación.
import string

# re.escape(): protege los signos, para que la expresión regular los tome tal cual.
signos_protegidos = re.escape(string.punctuation)
# "[...]" en una expresión regular: cualquiera de los caracteres de adentro.
patron_de_signos = "[" + signos_protegidos + "]"


def contar_como_text_mining(textos):
    """Cuenta las palabras de más de 3 letras, limpiando a mano como en text mining."""
    palabras = []
    for texto in textos:
        limpio = texto.lower()
        # re.sub(patrón, reemplazo, texto): cambia cada signo por un espacio
        limpio = re.sub(patron_de_signos, " ", limpio)
        for palabra in limpio.split():
            if len(palabra) > 3:
                palabras.append(palabra)
    return Counter(palabras)


conteo_viejo = contar_como_text_mining(muestra)
# most_common(15): las 15 más frecuentes, de mayor a menor
quince_viejas = conteo_viejo.most_common(15)
print(quince_viejas)

In [ ]:
# --- Camino 2: con spaCy ---

def contar_con_spacy(textos):
    """Cuenta los lemas de sustantivos comunes y propios que no son palabras vacías."""
    componentes_apagados = ["parser", "ner"]  # disable=: no los necesitamos acá
    lemas = []
    for documento in nlp.pipe(textos, batch_size=50, disable=componentes_apagados):
        for token in documento:

            # Solo nos interesan los tokens que son palabras...
            if token.is_alpha:

                # ...que no sean palabras vacías...
                if not token.is_stop:

                    # ...y que sean sustantivos comunes o propios.
                    if token.pos_ in ("NOUN", "PROPN"):
                        lema = token.lemma_.lower()
                        lemas.append(lema)
    return Counter(lemas)


conteo_nuevo = contar_con_spacy(muestra)
quince_nuevas = conteo_nuevo.most_common(15)
print(quince_nuevas)

In [ ]:
# Los dos rankings, lado a lado.
viejos = conteo_viejo.most_common(15)
nuevos = conteo_nuevo.most_common(15)

filas = []
# posicion: el lugar en los dos rankings a la vez (0 es el primero)
for posicion in range(15):
    palabra_vieja, n_viejo = viejos[posicion]
    palabra_nueva, n_nuevo = nuevos[posicion]
    puesto = posicion + 1
    fila = {
        "puesto": puesto,
        "text_mining": palabra_vieja,
        "veces": n_viejo,
        "spacy_lema_pos": palabra_nueva,
        "veces_spacy": n_nuevo,
    }
    filas.append(fila)

pd.DataFrame(filas)

En la columna izquierda quedan palabras que hay que sacar a mano: conectores,
verbos vacíos, adverbios. Es exactamente el trabajo que en text mining hiciste
mirando el top y agregando entradas a `add_stop_words`.

En la derecha no hubo lista a mano. El filtro fue `token.pos_ in ("NOUN", "PROPN")`:
en vez de decir qué palabras no querés, decís **qué categoría gramatical querés**.

Y hay una diferencia más de fondo: la columna derecha cuenta **lemas**, así que
*año* y *años* son la misma entrada.

## 5. Ejercicio

1. Cambiá el filtro de POS en `contar_con_spacy`. Probá tres versiones:
   - solo `NOUN` → temas
   - solo `VERB` → acciones
   - solo `ADJ` → valoraciones

   Los tres son análisis distintos del mismo corpus, no tres versiones del mismo.

2. Tomá los 400 párrafos de **2015** en lugar de los primeros 400 y compará el
   ranking de sustantivos con el de 2004. ¿Cambia el tema?

3. Sobre un párrafo cualquiera, extraé todos los pares sujeto-verbo con `dep_`.
   ¿Quién actúa en ese texto?

In [ ]:
# TU CÓDIGO ACÁ
#
# Paso 1: copiá acá contar_con_spacy, ponele otro nombre (por ejemplo
#         contar_verbos) y cambiá la condición del filtro por
#             if token.pos_ == "VERB":
#         Hacé otra copia para "ADJ".
#
# Paso 2: para 2015, primero armá la máscara y después filtrá:
#             es_de_2015 = corpus["anio"] == 2015
#             parrafos_2015 = corpus[es_de_2015]
#             primeros_2015 = parrafos_2015.head(400)
#             textos_2015 = list(primeros_2015["texto"])
#         (2015 tiene 281 párrafos: head(400) te los da todos.)
#
# Paso 3: tomá un párrafo, por ejemplo corpus.loc[10, "texto"], procesalo con
#         nlp y recorré sus tokens como en la sección 1: si token.dep_ es
#         "nsubj", su verbo es token.head.

> ### Perilla
>
> Toda esta comparación se hizo sobre 400 párrafos, que son los primeros del
> archivo — o sea, casi todo 2004.
>
> Cambiá el tamaño de la muestra y volvé a correr. Con 100 párrafos, ¿el ranking
> es el mismo? ¿A partir de cuántos párrafos se estabiliza?
>
> No hay un número correcto. Lo que hay es una obligación: **decir cuál usaste**
> y por qué. Un ranking de palabras sin el tamaño de la muestra al lado no es un
> resultado, es una anécdota.

In [ ]:
for cuantos in [50, 100, 400]:
    primeros = corpus.head(cuantos)
    textos = list(primeros["texto"])
    conteo = contar_con_spacy(textos)
    primeras = []
    for palabra, veces in conteo.most_common(6):
        primeras.append(palabra)
    primeras_como_texto = ", ".join(primeras)
    print(cuantos, "párrafos:", primeras_como_texto)

## Lo que hiciste hoy

- Leíste un árbol de dependencias y extrajiste sujetos y modificadores.
- Usaste grupos nominales.
- Detectaste entidades y las pasaste a una tabla de pandas.
- Rehiciste el conteo de palabras clave de text mining con lemas y POS, y viste
  la diferencia.
- Viste que el NER produce errores que hay que mirar.

## Lo que viene

Con esto termina el primer encuentro. En el `04` empieza el segundo, y arranca
justamente por ahí: **qué modelo estamos usando, qué se arregla si lo cambiamos,
y qué no se arregla nunca.**